In [ ]:
# A DuckDB relation as the source: every request the grid makes is a SQL statement on `con`.
import duckdb
import pandas as pd
from lattice_grid_jupyter import LatticeGridWidget

con = duckdb.connect()
con.execute("CREATE TABLE events AS SELECT i AS id, 'user ' || (i % 50) AS who, i % 7 AS day, "
            "(i * 13) % 1000 AS amount FROM range(5000) t(i)")
w = LatticeGridWidget(con.table("events"), offline=True, windowed=True,
                      state={"version": 2,
                             "sort": [{"col": "amount", "dir": "desc"}],
                             "filters": {"col": "amount", "op": "gte", "value": 500}})
print("MODE=%s" % ("windowed" if w.windowed else "client"))
print("ENGINE=%s" % w._engine.name)
print("BEFORE_COLUMNS=%s" % list(w.view.columns))

In [ ]:
w

In [ ]:
# Select two rows through the grid's public selection API.
from IPython.display import Javascript, display
display(Javascript("window.__latticeGrids.grids.get(%r).grid.selection.set(['3', '5'])" % w._uid))

In [ ]:
# A later cell reads what the grid reported.
print("SELECTED_FIRST_READ=%s" % list(w.selected["id"]))

In [ ]:
# New schema: `amount` is gone, `score` arrives.
con.execute("CREATE TABLE events2 AS SELECT i AS id, 'user ' || (i % 50) AS who, i % 7 AS day, "
            "i % 3 AS score FROM range(300) t(i)")
w.set_data(con.table("events2"))

In [ ]:
assert w.state.get("sort") in (None, []), w.state
assert not w.state.get("filters"), w.state
cols = list(w.view.columns)
assert "amount" not in cols and "score" in cols, cols
assert len(w.view) == 300
print("AFTER_COLUMNS=%s" % cols)
print("AFTER_ROWS=%d" % len(w.view))
print("AFTER_SORT=%s FILTERS=%s" % (w.state.get("sort"), w.state.get("filters")))

In [ ]:
# Read the selection again, in a later cell: the grid reports it between cells.
print("SELECTED=%s" % list(w.selected["id"]))